In [15]:
import copy
import random
import string

In [18]:
NUM_PRODUCTS = 1000000
NUM_RATINGS = 100
NUM_SIMULATIONS = 1000

products_ids = list(range(NUM_PRODUCTS))
ratings_ids  = list(range(NUM_RATINGS))

Products = {
    pid: {"name": f"P{pid}", "price": 1, "quantity": 10, "sold": 0, "seller": 1}
    for pid in products_ids
}

Ratings = {
    rid: {"productId": 0, "score": 3, "comment": "Good"}
    for rid in ratings_ids
}

COMMENTS = ["Good", "Bad", "Excellent", "Average", "Poor"]

def product_invariant(p):
    return p["price"] >= 0 and p["quantity"] >= 0 and p["sold"] >= 0 and p["name"] != ""

def rating_invariant(r):
    return 1 <= r["score"] <= 5 and r["comment"] != "" and r["productId"] in products_ids

def buy_product(pid, qty):
    assert pid in Products, f"Product with id={pid} does not exist"
    state = copy.deepcopy(Products[pid])
    if state["quantity"] >= qty:
        state["quantity"] -= qty
        state["sold"] += qty
    assert product_invariant(state), f"Invariant violated for product id={pid}"
    return state

def update_product(pid, new_name, new_price, new_qty):
    assert pid in Products, f"Product with id={pid} does not exist"
    state = copy.deepcopy(Products[pid])
    state["name"] = new_name
    state["price"] = new_price
    state["quantity"] = new_qty
    assert product_invariant(state), f"Invariant violated for product id={pid}"
    return state

def delete_product(pid):
    assert pid in Products, f"Product with id={pid} does not exist"
    state = copy.deepcopy(Products[pid])
    state["quantity"] = 0
    state["sold"] = 0
    assert product_invariant(state), f"Invariant violated for product id={pid}"
    return state

def add_rating(rid, pid, score):
    assert pid in Products, f"Product with id={pid} does not exist"
    comment = random.choice(COMMENTS)
    state = {"productId": pid, "score": score, "comment": comment}
    assert rating_invariant(state), f"Invariant violated for rating id={rid} on product id={pid}"
    return state


violations = 0

operations = [
    lambda pid, new_name, new_price, new_qty, qty, rid, score: buy_product(pid, qty),
    lambda pid, new_name, new_price, new_qty, qty, rid, score: update_product(pid, new_name, new_price, new_qty),
    lambda pid, new_name, new_price, new_qty, qty, rid, score: delete_product(pid),
    lambda pid, new_name, new_price, new_qty, qty, rid, score: add_rating(rid, pid, score)
]

for _ in range(NUM_SIMULATIONS):
    pid = random.choice(products_ids)
    new_name = ''.join(random.choices(string.ascii_uppercase, k=5))
    qty = random.randint(0, 5)
    new_price = random.randint(0, 10)
    new_qty = random.randint(0, 10)
    rid = random.choice(ratings_ids)
    score = random.randint(1, 5)

    random.shuffle(operations)
    try:
        for op in operations:
            op(pid, new_name, new_price, new_qty, qty, rid, score)
    except AssertionError:
        violations += 1


if violations == 0:
    print("✔ All invariants preserved in simulation.")
else:
    print(f"⚠ Invariant violations detected in {violations} operations.")

✔ All invariants preserved in simulation.
